# Cold Diffusion Pipeline

This notebook runs setup, training, generation, and testing of the Cold Diffusion re-implementation codebase.
Use the YAML configuration files in `configs/` for different diffusion scenarios.

## 1. Setup & Environment

Mount Google Drive (if running on Colab) and navigate to the project directory. The cell automatically detects your environment and installs necessary dependencies.

In [ ]:
import os
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# --- Configuration: Path to your project on Google Drive ---
USE_GOOGLE_DRIVE = IN_COLAB 
DRIVE_PROJECT_DIR = "DRIVE PROJECT DIRECTORY PATH HERE"

if IN_COLAB and USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
    root = Path(DRIVE_PROJECT_DIR).resolve()
    if not (root / "train.py").exists():
        print(f"⚠️ Warning: train.py not found under {root}. Check DRIVE_PROJECT_DIR.")
    os.chdir(root)
    !pip install -q -r requirements.txt
elif IN_COLAB:
    # Running in Colab without mounting Drive
    os.chdir("/content")
    !pip install -q -r requirements.txt
else:
    # Running locally
    os.chdir(Path.cwd().resolve())

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("Working directory:", ROOT)

## 2. Load Configuration

Load the YAML configuration file. If you are on Colab, we override the dataset download path to point to `/content/data`.

In [ ]:
import utils
import train
import generate

print(f"Active Device: {utils.get_device()}")

# Select your configuration file here (e.g., configs/mnist_blur.yaml, configs/mnist_pixelate.yaml, configs/cifar10.yaml, configs/cifar10_pixelate.yaml)
CONFIG_PATH = "configs/mnist_blur.yaml"
config = utils.load_config(CONFIG_PATH)

# Colab Optimization: Download datasets to local VM storage to avoid Drive I/O bottlenecks
if IN_COLAB:
    config['dataset']['root'] = '/content/data'

print(f"\nSuccessfully loaded configuration for: {config['dataset']['name'].upper()}")
print(f"Degradation Type: {config['degradation']['type']}")
print(f"Timesteps: {config['degradation']['timesteps']}")

## 3. Train Model

Run the training loop. All hyperparameters, model architecture specs, and checkpoint saving logic are read directly from the configuration dictionary.

In [ ]:
# If you want to resume training from a checkpoint, pass the path string to resume_path.
# Example: train.train(config, resume_path="outputs/cifar10/blur/checkpoint_epoch_50.pt")

train.train(config, resume_path=None)

## 4. Generate & Restore

Test the trained model by applying maximum degradation to real test images and running the reverse sampler to restore them. You can switch the algorithm here depending on what sampler you want to evaluate.

In [ ]:
# Algorithms available: 'sampler' (default Alg 2), 'direct', 'adafusion', 'projection'
generate.generate(config, algorithm="sampler")

## 5. View Results

Display the resulting image grids inline from the `samples/` directory generated in the previous step.

In [ ]:
from IPython.display import Image, display

print("Final Pipeline Results")
primary_paths = [
    ("1. Ground Truth", "samples/ground_truth.png"),
    ("2. Starting State (x_T)", "samples/starting_state_xT.png"),
    ("3. Final Restoration (x_0)", "samples/generated_grid_final.png"),
]

for title, path in primary_paths:
    if Path(path).exists():
        print(f"\n{title}")
        display(Image(filename=path))
    else:
        print(f"\n[Missing] {title} not found at {path}")

print("\nModel Progression Snapshots (x0_hat)")
x0_dir = Path("samples/x0_predictions")
if x0_dir.exists():
    x0_frames = sorted(x0_dir.glob("x0_hat_step_*.png"))
    if x0_frames:
        indices = [0, len(x0_frames) // 2, -1]
        for idx in indices:
            frame = x0_frames[idx]
            print(f"\n{frame.name}")
            display(Image(filename=str(frame)))
else:
    print(f"No intermediate progression frames found in {x0_dir}")